In [1]:
import joblib
import pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.linear_model import SGDClassifier, SGDRegressor, LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor 
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import time
import pytz
import json
import os

In [2]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-09-08_59_15_PM'

In [3]:
experiment_config = {
    "experiment": {
        "id": f"{dt_str}_catboost",
        "model": "catboost",
        "type": "baseline",
        "dataset_type": "features",
        "description": "basic catboost baseline",
        "target": "classification",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "params": {
        "cat_features": ["gender"],
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "custom_metric": ["AUC"],
        "n_estimators": 1100,
        "learning_rate": 0.1,
        "max_depth": 6,
        "early_stopping_rounds": 10
    },

    "fit_params": {
    }
}

In [4]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)
experiment_path

PosixPath('/kaggle/working/experiments/2026-08-09-08_59_15_PM_catboost')

In [5]:
with open(experiment_path / "config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

In [6]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [7]:
raw_train_id = pd.read_csv(f"{data_path}/raw/train.csv")["id"]
X = pd.read_csv(f"{data_path}/processed/train_{experiment_config["experiment"]["dataset_type"]}.csv")
X_test = pd.read_csv(f"{data_path}/processed/test_{experiment_config["experiment"]["dataset_type"]}.csv")
y = pd.read_csv(f"{data_path}/processed/train_labels.csv")

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 23 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               662440 non-null  float64
 1   daily_screen_time_hours           595515 non-null  float64
 2   social_media_hours                557374 non-null  float64
 3   gaming_hours                      564548 non-null  float64
 4   work_study_hours                  639851 non-null  float64
 5   sleep_hours                       646889 non-null  float64
 6   notifications_per_day             623785 non-null  float64
 7   app_opens_per_day                 610659 non-null  float64
 8   weekend_screen_time               579306 non-null  float64
 9   gender                            662335 non-null  object 
 10  stress_level                      636221 non-null  float64
 11  academic_work_impact              647145 non-null  f

In [8]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        if experiment_config["experiment"]["model"] == "catboost":
            frame[col] = frame[col].fillna('Missing')
        
        frame[col] = frame[col].astype('category')

In [9]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 23 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            691369 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              64714

In [10]:
def make_model(config):
    name = config["experiment"]["model"]
    target = config["experiment"]["target"]
    params = config["params"]

    model_dict = {
        "adaboost": (AdaBoostClassifier, AdaBoostRegressor),
        "gradientboost": (GradientBoostingClassifier, GradientBoostingRegressor),
        "catboost": (CatBoostClassifier, CatBoostRegressor),
        "xgboost": (XGBClassifier, XGBRegressor),
        "lightgbm": (LGBMClassifier, LGBMRegressor),
        "randomforest": (RandomForestClassifier, RandomForestRegressor),
        "extratrees": (ExtraTreesClassifier, ExtraTreesRegressor),
        "hgbc": (HistGradientBoostingClassifier, HistGradientBoostingRegressor),
        "knn": (KNeighborsClassifier, KNeighborsRegressor),
        "sgd": (SGDClassifier, SGDRegressor),
        "linear": (LogisticRegression, LinearRegression),
        "decisiontree": (DecisionTreeClassifier, DecisionTreeRegressor),
        "mlp": (MLPClassifier, MLPRegressor),
    }

    target_index = target == "regression"
    model_class = model_dict[name][target_index]

    return model_class(**params)

In [11]:
def oof_fit(config, model, X_train, y_train, X_valid, y_valid):
    name = config["experiment"]["model"]
    no_eval_models = ["adaboost", "gradientboost", "randomforest", "extratrees", "knn", "sgd", "linear", "decisiontree", "mlp"]

    if name in no_eval_models:
        model.fit(X_train, y_train)
    elif name == "hgbc":
        model.fit(X_train, y_train, X_val=X_valid, y_val=y_valid)
    else:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

In [12]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

y_cv = pd.Series(index=y.index, dtype=float, name=target_column)
fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = make_model(experiment_config)
    oof_fit(experiment_config, model, X_train, y_train, X_valid, y_valid)

    y_pred = model.predict_proba(X_valid)[:, 1]

    y_cv.iloc[valid_index] = y_pred

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(round(fold_auc_score, 5))

elapsed = time.time() - start_time

y_pred_df = pd.concat([raw_train_id, y_cv], axis=1)
y_pred_df.to_csv(experiment_path / "oof.csv", index=False)

0:	test: 0.9045839	best: 0.9045839 (0)	total: 382ms	remaining: 7m
1:	test: 0.9103248	best: 0.9103248 (1)	total: 673ms	remaining: 6m 9s
2:	test: 0.9142448	best: 0.9142448 (2)	total: 944ms	remaining: 5m 45s
3:	test: 0.9146252	best: 0.9146252 (3)	total: 1.24s	remaining: 5m 40s
4:	test: 0.9180987	best: 0.9180987 (4)	total: 1.53s	remaining: 5m 35s
5:	test: 0.9192188	best: 0.9192188 (5)	total: 1.82s	remaining: 5m 31s
6:	test: 0.9203702	best: 0.9203702 (6)	total: 2.11s	remaining: 5m 29s
7:	test: 0.9216147	best: 0.9216147 (7)	total: 2.38s	remaining: 5m 24s
8:	test: 0.9227597	best: 0.9227597 (8)	total: 2.67s	remaining: 5m 23s
9:	test: 0.9229457	best: 0.9229457 (9)	total: 2.94s	remaining: 5m 20s
10:	test: 0.9237220	best: 0.9237220 (10)	total: 3.23s	remaining: 5m 19s
11:	test: 0.9246186	best: 0.9246186 (11)	total: 3.52s	remaining: 5m 19s
12:	test: 0.9252437	best: 0.9252437 (12)	total: 3.78s	remaining: 5m 15s
13:	test: 0.9259813	best: 0.9259813 (13)	total: 4.08s	remaining: 5m 16s
14:	test: 0.92650

In [13]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

Validation AUC: 0.9620227383783632


In [14]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "dataset_type": f"{experiment_config["experiment"]["dataset_type"]}",
    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "random_state": 0,
        "fold_scores": fold_scores,
        "mean": round(sum(fold_scores) / len(fold_scores), 5),
        "std": round(float(pd.Series(fold_scores).std(ddof=1)), 5)
    },
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    },
    "training": {
        "duration_seconds": round(elapsed, 2)
    }
}

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [15]:
best_iteration = model.best_iteration if hasattr(model, "best_iteration") else experiment_config["params"]["n_estimators"]

experiment_config["params"]["early_stopping_rounds"] = None
experiment_config["params"]["n_estimators"] = best_iteration

model = make_model(experiment_config)
model.fit(X, y)

joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")

0:	total: 339ms	remaining: 6m 12s
1:	total: 643ms	remaining: 5m 52s
2:	total: 948ms	remaining: 5m 46s
3:	total: 1.27s	remaining: 5m 49s
4:	total: 1.59s	remaining: 5m 49s
5:	total: 1.93s	remaining: 5m 52s
6:	total: 2.25s	remaining: 5m 51s
7:	total: 2.58s	remaining: 5m 52s
8:	total: 2.9s	remaining: 5m 51s
9:	total: 3.22s	remaining: 5m 50s
10:	total: 3.52s	remaining: 5m 48s
11:	total: 3.84s	remaining: 5m 48s
12:	total: 4.14s	remaining: 5m 46s
13:	total: 4.46s	remaining: 5m 46s
14:	total: 4.78s	remaining: 5m 46s
15:	total: 5.1s	remaining: 5m 45s
16:	total: 5.45s	remaining: 5m 47s
17:	total: 5.77s	remaining: 5m 46s
18:	total: 6.08s	remaining: 5m 45s
19:	total: 6.38s	remaining: 5m 44s
20:	total: 6.69s	remaining: 5m 43s
21:	total: 7.01s	remaining: 5m 43s
22:	total: 7.33s	remaining: 5m 43s
23:	total: 7.66s	remaining: 5m 43s
24:	total: 7.98s	remaining: 5m 43s
25:	total: 8.32s	remaining: 5m 43s
26:	total: 8.63s	remaining: 5m 43s
27:	total: 8.96s	remaining: 5m 43s
28:	total: 9.28s	remaining: 5m 4

['/kaggle/working/experiments/2026-08-09-08_59_15_PM_catboost/catboost.pkl']

In [16]:
y_pred = model.predict_proba(X_test)[:, 1]
ss[target_column] = y_pred

ss.to_csv(experiment_path / f"{experiment_config["experiment"]["model"]}_submission.csv", index=False)
ss

,id,addicted_label
0,691369,0.999383
1,691370,0.944308
2,691371,0.943360
3,691372,0.983831
4,691373,0.998016
...,...,...
296297,987666,0.999997
296298,987667,0.919033
296299,987668,0.181456
296300,987669,0.635690
